<a href="https://colab.research.google.com/github/RAHULRAJPAL75/Rahul_IN226051102_NLP/blob/main/Task_3_Build_a_Chatbot_using_Hugging_Face_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install ipywidgets --upgrade
!jupyter nbextension enable --py widgetsnbextension

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 62.4 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


Enabling notebook extension jupyter-js-widgets/extension...
      - Validating: OK


In [ ]:
!pip install transformers torch


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [ ]:
model_name = "microsoft/DialoGPT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/351M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

In [ ]:
def rule_based_response(user_input):
    """
    This function handles common queries with fixed responses
    to improve chatbot accuracy.
    """
    user_input = user_input.lower()

    if "hello" in user_input:
        return "Hello! Nice to meet you. How can I assist you today?"

    elif "what is artificial intelligence" in user_input or "what is ai" in user_input:
        return "Artificial Intelligence refers to the simulation of human intelligence by machines."

    elif "who created python" in user_input:
        return "Python was created by Guido van Rossum in 1991."

    elif "thank" in user_input:
        return "You're welcome! Feel free to ask more questions."

    return None

In [ ]:
def generate_response(user_input, chat_history_ids):
    """
    Generates response using DialoGPT transformer model.
    """
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')
    bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1) if chat_history_ids is not None else new_input_ids
    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)
    chat_history_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_length=300,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=False
    )
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response, chat_history_ids

In [ ]:
def chatbot():
    """
    Main chatbot loop for continuous conversation.
    """
    print("🤖 Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.\n")

    chat_history_ids = None

    while True:
        user_input = input("You: ")

        if user_input.lower() in ["exit", "quit"]:
            print("🤖 Chatbot: Goodbye! Have a great day!")
            break
        rule_response = rule_based_response(user_input)

        if rule_response:
            print("🤖 Chatbot:", rule_response)
            continue
        response, chat_history_ids = generate_response(user_input, chat_history_ids)

        print("🤖 Chatbot:", response)

In [ ]:
chatbot()

🤖 Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.

You: Hello
🤖 Chatbot: Hello! Nice to meet you. How can I assist you today?
You: What is AI?
🤖 Chatbot: Artificial Intelligence refers to the simulation of human intelligence by machines.
You: Who created Python?
🤖 Chatbot: Python was created by Guido van Rossum in 1991.
You: Tell me about space
🤖 Chatbot: I'm a space pirate.
You: exit
🤖 Chatbot: Goodbye! Have a great day!
